<div style="max-width:100%;box-sizing:border-box;overflow:visible;border-top:4px solid #0f766e;padding:32px 0 20px;margin:0 0 24px">
  <div style="display:block;color:#0f766e;font-size:13px;line-height:1.8;font-weight:700;letter-spacing:0.8px;text-transform:uppercase;margin:0 0 8px">LAB 10 · DATA WAREHOUSING WITH APACHE DORIS</div>
  <div style="color:#17212b;font-size:30px;line-height:1.3;font-weight:750;margin:0 0 10px">Build and Reconcile a Service Table, and Measure Payments</div>
</div>

A metric is a contract before it is SQL. You write down the contract of daily orders and order amount, build the service table `daily_order_metrics_l2` at the grain of order date and region from Lab 9's order fact, and reconcile it with the detail, with a full outer join that catches a day that a service table lost and an inner join misses. On the real days and regions you see a cumulative window keep its grain and an average of averages go wrong. Then you measure the case's payments: paid GMV in a window, the conversion of the orders created in it, and repeat buyers; compare exact and approximate distinct counts; and check that a smaller table answers like the detail while it reads fewer rows. Plan about 90 minutes.

**Module 10: Metric Processing and Delivery** · [Lesson](course.md) · [Quiz](quiz10_metric_processing.ipynb) · [Course README](../../README.md)

## Before you begin

Complete Lab 9 first, in the same database. This Lab reads its order fact `orders_fact_l2` and the case's `dwd_orders_l2`, `dwd_payments_l2`, and `customers_case_dim_l2`, and changes none of them; if one is missing, the setup cell stops and says so.

The Lab rebuilds `daily_order_metrics_l2`, which Lab 11 publishes; for the case, `dwd_refunds_l2`, `paid_order_metrics_l2`, and the view `bi_paid_kpis_l2`, which Lab 11's optional dashboard reads; and for the measurement of section 8, `perf_orders_detail_l2` and `perf_daily_orders_l2`. All of them stay in your lab database. Everything runs in the course sandbox.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, normalized
from dw_course.ui import show_records, show_sql
from dw_course.warehouse import case_fixture, compare_query_profiles

lab = connect_sandbox(module=10)

## 1. Write the metric contract, then build the service table

"Daily orders" needs answers before it needs SQL. The contract of this Lab:

| Item | Daily orders and order amount |
| --- | --- |
| Population | The orders of `orders_fact_l2` that are not CANCELLED |
| Grain | One row per order date and region |
| Measures | `order_count`, the number of orders; `gross_amount`, the sum of `order_amount`, the orders' pre-tax value, not money received |
| Time | `order_date`, the date of the order's version 1 event, in the session's time zone, +08:00 |
| Ratio | Average order amount = `gross_amount` / `order_count`, computed from the sums at the grain asked for; NULL when there are no orders |

The service table `daily_order_metrics_l2` stores the measures at that grain. It is an Aggregate Key table: rows with the same key are added up, so `order_count` and `gross_amount` keep adding as long as rows arrive. That is right for new orders and wrong for the same batch loaded twice, so the cell rebuilds the table from the detail each time it runs.

In [ ]:
lab.execute("DROP TABLE IF EXISTS daily_order_metrics_l2")
service_ddl = """CREATE TABLE daily_order_metrics_l2 (
    order_date DATE NOT NULL,
    region VARCHAR(16) NOT NULL,
    order_count BIGINT SUM NOT NULL DEFAULT "0",
    gross_amount DECIMAL(18,2) SUM NOT NULL DEFAULT "0.00"
)
AGGREGATE KEY(order_date, region)
DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1")"""
service_load = """INSERT INTO daily_order_metrics_l2
SELECT order_date, region, COUNT(*), SUM(order_amount)
FROM orders_fact_l2
WHERE status <> 'CANCELLED'
GROUP BY order_date, region"""
show_sql("The service table and its load", service_ddl + ";\n\n" + service_load + ";")
lab.execute(service_ddl)
lab.execute(service_load)
lab.sql("SELECT * FROM daily_order_metrics_l2 ORDER BY order_date, region", title="The service table")
expect(lab.query("SELECT CAST(order_date AS STRING), region, order_count, gross_amount FROM daily_order_metrics_l2 ORDER BY order_date, region"),
       [("2026-01-01", "EAST", 5, "720.00"), ("2026-01-01", "WEST", 4, "480.00"), ("2026-01-02", "EAST", 1, "110.00")],
       title="Three rows at the grain of the contract")

**Expected result**

The definition and the load, then three rows: 2026-01-01 EAST with 5 orders and 720.00, 2026-01-01 WEST with 4 and 480.00, and 2026-01-02 EAST with 1 and 110.00, and a green check. Together 10 orders and 1310.00, the orders of Lab 8's view.

The table stores counts and sums, not averages: an average stored per row could not be added up to a day or to the whole period. Loading the same batch twice would show 10 orders and 1440.00 on January 1 EAST; the cell drops and rebuilds the table instead.

## 2. Roll the metric up for a dashboard

A dashboard shows one row per day. It adds up the service rows of each date and computes the average from the sums, so the average stays right however many regions a day has. A conditional sum shows one region beside the total in the same row: `SUM(CASE WHEN region = 'EAST' THEN order_count ELSE 0 END)` adds only the EAST rows of each group.

In [ ]:
dashboard_sql = """SELECT CAST(order_date AS STRING) AS order_date,
       SUM(CASE WHEN region = 'EAST' THEN order_count ELSE 0 END) AS east_orders,
       SUM(order_count) AS all_orders,
       SUM(gross_amount) AS gross_amount,
       CAST(SUM(gross_amount) / SUM(order_count) AS DECIMAL(18,2)) AS average_order_amount
FROM daily_order_metrics_l2
GROUP BY order_date
ORDER BY order_date"""
lab.sql(dashboard_sql, title="One row per day, for a dashboard")
expect(lab.query(dashboard_sql), [("2026-01-01", 5, 9, "1200.00", "133.33"), ("2026-01-02", 1, 1, "110.00", "110.00")],
       title="The daily rows and their averages")

**Expected result**

Two rows: 2026-01-01 with 5 EAST orders of 9, 1200.00, and an average of 133.33; 2026-01-02 with 1 EAST order of 1, 110.00, and 110.00. A green check.

1200.00 / 9 is 133.333…; the `CAST` rounds it for display. A consumer who needs the exact value takes the sums, not the rounded average.

## 3. Reconcile the service table with the detail

A service table is right when the detail, computed again and independently, gives the same numbers. The query below computes the contract from `orders_fact_l2` (`detail`) and reads the service table (`serving`), then joins them on the service grain, date and region. A `FULL OUTER JOIN` keeps a row that one side has and the other lacks, with `NULL` on the missing side.

The cell runs the comparison three times: with the full outer join on the complete service table; then twice with a service table that lacks January 2, as if that day's load had failed: once with an inner join, once with the full outer join. Predict which of the last two notices the missing day.

In [ ]:
reconcile_sql = """WITH detail AS (
    SELECT order_date, region, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
    FROM orders_fact_l2
    WHERE status <> 'CANCELLED'
    GROUP BY order_date, region
), serving AS (
    SELECT order_date, region, SUM(order_count) AS order_count, SUM(gross_amount) AS gross_amount
    FROM daily_order_metrics_l2 {lost_day}
    GROUP BY order_date, region
)
SELECT CAST(COALESCE(d.order_date, s.order_date) AS STRING) AS order_date,
       COALESCE(d.region, s.region) AS region,
       d.order_count AS detail_orders, s.order_count AS serving_orders,
       d.gross_amount AS detail_amount, s.gross_amount AS serving_amount
FROM detail d
{join} serving s ON s.order_date = d.order_date AND s.region = d.region
ORDER BY order_date, region"""
complete = reconcile_sql.format(lost_day="", join="FULL OUTER JOIN")
lab.sql(complete, title="Detail and service table, full outer join")
expect(lab.query(complete), [("2026-01-01", "EAST", 5, 5, "720.00", "720.00"), ("2026-01-01", "WEST", 4, 4, "480.00", "480.00"),
                             ("2026-01-02", "EAST", 1, 1, "110.00", "110.00")], title="The service table matches the detail")
lost = "WHERE order_date < '2026-01-02'"
inner = reconcile_sql.format(lost_day=lost, join="INNER JOIN")
outer = reconcile_sql.format(lost_day=lost, join="FULL OUTER JOIN")
lab.sql(inner, title="A service table without January 2, inner join")
lab.sql(outer, title="A service table without January 2, full outer join")
expect([row for row in lab.query(inner) if row[2] != row[3] or row[4] != row[5]], [],
       title="The inner join finds no difference")
expect([row[:2] for row in lab.query(outer) if row[3] is None], [("2026-01-02", "EAST")],
       title="The full outer join shows the missing day")

**Expected result**

The full outer join on the complete table: three rows, each with the same orders and amount on both sides, and a green check. Without January 2, the inner join shows two rows, both equal, and the full outer join three, the last one 2026-01-02 EAST with 1 order and 110.00 in the detail and `NULL` in the service table. Two green checks: the inner join finds no difference, and the full outer join shows the missing day.

The inner join kept only the dates that both sides have, so a day that the service table lost disappeared from the comparison instead of failing it. Keep the `NULL` until you know why it is there: a missing row is not a day with zero orders.

## 4. Windows and ratios on the real days and regions

A window function computes over related rows and keeps each row: `SUM(gross_amount) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)` adds up the amounts from the first day to the current one, and `LAG(gross_amount) OVER (ORDER BY order_date)` takes the previous row's amount. The CTE first makes one row per day, so the windows run on days, not on orders or regions.

The query also computes January 1's average twice: the average of the two regions' averages, and the day's sum divided by its count. Predict which one is the day's average order amount.

In [ ]:
window_sql = """WITH by_region AS (
    SELECT order_date, region, SUM(order_count) AS order_count, SUM(gross_amount) AS gross_amount
    FROM daily_order_metrics_l2 GROUP BY order_date, region
), daily AS (
    SELECT order_date,
           SUM(order_count) AS order_count,
           SUM(gross_amount) AS gross_amount,
           CAST(AVG(gross_amount / order_count) AS DECIMAL(18,2)) AS average_of_region_averages
    FROM by_region GROUP BY order_date
)
SELECT CAST(order_date AS STRING) AS order_date, order_count, gross_amount,
       average_of_region_averages,
       CAST(gross_amount / order_count AS DECIMAL(18,2)) AS average_order_amount,
       SUM(gross_amount) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_amount,
       LAG(gross_amount) OVER (ORDER BY order_date) AS previous_day_amount
FROM daily
ORDER BY order_date"""
lab.sql(window_sql, title="Days, their averages, and two windows")
expect(lab.query(window_sql), [("2026-01-01", 9, "1200.00", "132.00", "133.33", "1200.00", None),
                               ("2026-01-02", 1, "110.00", "110.00", "110.00", "1310.00", "1200.00")],
       title="The windows keep one row per day; the average of averages is not the day's average")

**Expected result**

Two rows and a green check. 2026-01-01: 9 orders, 1200.00, an average of the region averages of 132.00, an average order amount of 133.33, 1200.00 so far, and no previous day (`NULL`). 2026-01-02: 1 order, 110.00, 110.00 for both averages, 1310.00 so far, and 1200.00 the day before.

On January 1, EAST averages 144.00 over 5 orders and WEST 120.00 over 4. Their simple average, 132.00, weighs the 4 WEST orders as much as the 5 EAST ones; the day's average weighs every order once: 1200.00 / 9 = 133.33. Store the sums and counts, and divide at the grain that you report. `LAG` took the previous row: here the previous day, because no day is missing; with a gap, it would be an older day.

## 5. From order amount to payment metrics

Order amount says what was ordered; it does not say what was paid. The case of Lab 9 sections 5–7 has the payments: its eight orders have seven payment deliveries, one of them a retry and one after the reporting cutoff, and one refund of part of a payment. The metrics below follow this contract; **gross merchandise value (GMV)** here means the value of the paid orders.

| Metric | Contract for this case |
| --- | --- |
| Paid gross merchandise value (GMV) | Sum deduplicated successful payment amounts in `[January 1, January 4)`, Asia/Shanghai; refunds are separate |
| Paid orders / customers | Distinct order / customer IDs in that payment population |
| Created-cohort conversion | Orders created in that window and paid before January 4, divided by all orders created in that window |
| Repeat-buyer rate | Customers with at least two paid orders in the payment window, divided by paid customers |
| Net collections | Paid GMV minus refunds received in the same window; not accounting profit |

The fixture deliberately has **one successful payment per order**. A split-payment system would need a different order-level reduction before counting paid orders. Latest order status is not a substitute for payment evidence: a refunded order still contributed its original payment.

The cell builds the refunds' table and `paid_order_metrics_l2`, one row per payment date, then checks the window's totals against the detail.

In [ ]:
case = case_fixture()
window_start, window_end, as_of = case["window_start"], case["window_end"], case["as_of"]
lab.execute("DROP VIEW IF EXISTS bi_paid_kpis_l2")
lab.execute("DROP TABLE IF EXISTS paid_order_metrics_l2")
lab.execute("DROP TABLE IF EXISTS dwd_refunds_l2")
lab.execute("""CREATE TABLE dwd_refunds_l2 (
    refund_id VARCHAR(32) NOT NULL, payment_id VARCHAR(32) NOT NULL,
    refund_amount DECIMAL(18,2) NOT NULL, refund_time DATETIME NOT NULL
) UNIQUE KEY(refund_id) DISTRIBUTED BY HASH(refund_id) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
lab.insert("dwd_refunds_l2", ["refund_id","payment_id","refund_amount","refund_time"], case["refunds"])
lab.execute("""CREATE TABLE paid_order_metrics_l2 (
    payment_date DATE NOT NULL, paid_orders BIGINT NOT NULL,
    paid_gmv DECIMAL(18,2) NOT NULL, paid_customers BIGINT NOT NULL
) DUPLICATE KEY(payment_date) DISTRIBUTED BY HASH(payment_date) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
lab.execute("""INSERT INTO paid_order_metrics_l2
SELECT DATE(p.payment_time), COUNT(DISTINCT p.order_id), SUM(p.amount), COUNT(DISTINCT o.customer_id)
FROM dwd_payments_l2 p JOIN dwd_orders_l2 o ON p.order_id=o.order_id
WHERE p.payment_time >= %s AND p.payment_time < %s
GROUP BY DATE(p.payment_time)""", (window_start, window_end))
expect(lab.query("SELECT CAST(payment_date AS STRING),paid_orders,paid_gmv,paid_customers FROM paid_order_metrics_l2 ORDER BY payment_date"), case["expected"]["payment_daily"],
       title="Three payment days, as the case expects")
# This is an independent detail calculation, not another read of the service result.
expect(lab.query("""SELECT COUNT(DISTINCT p.order_id), SUM(p.amount), COUNT(DISTINCT o.customer_id)
FROM dwd_payments_l2 p JOIN dwd_orders_l2 o ON p.order_id=o.order_id
WHERE p.payment_time >= %s AND p.payment_time < %s""", (window_start,window_end)), [(5,"660.00",3)],
       title="The window from the detail: 5 paid orders, 660.00, 3 customers")
lab.sql("SELECT * FROM paid_order_metrics_l2 ORDER BY payment_date", title="Paid orders per payment day");

**Expected result**

Two green checks, then three payment days: 2026-01-01 with 1 paid order, 100.00, and 1 customer; 2026-01-02 with 2, 270.00, and 2; 2026-01-03 with 2, 290.00, and 1.

The window has 5 paid orders and 660.00 from 3 customers. The retry of PAY-B counts once, because `dwd_payments_l2` keeps one row per payment ID, and PAY-E, paid on January 5, is outside the window. The daily customer counts add up to 4, not 3: customer 101 paid on two days.

## 6. Reconcile the window, not a sum of daily ratios

**Predict:** daily distinct-buyer counts sum to four, but the window has only three distinct buyers. One customer appears on two dates. Likewise, a payment on January 2 may belong to an order created January 1, so a same-day payment count divided by a same-day creation count is not cohort conversion.

Build a one-row consumer contract from detail. It keeps numerators and denominators, plus the explicit cutoff. January 5's payment is excluded; a partial refund reduces net collections, not paid GMV. Lab 11's optional dashboard reads this view.

In [ ]:
# Timestamp strings come from the committed case fixture, not user-supplied SQL.
lab.execute(f"""CREATE VIEW bi_paid_kpis_l2 AS
WITH paid AS (
    SELECT p.order_id, o.customer_id, p.amount AS amount
    FROM dwd_payments_l2 p JOIN dwd_orders_l2 o ON p.order_id=o.order_id
    WHERE p.payment_time >= '{window_start}' AND p.payment_time < '{window_end}'
), buyer_orders AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS paid_orders FROM paid GROUP BY customer_id
), cohort AS (
    SELECT o.order_id, MAX(CASE WHEN p.payment_time < '{as_of}' THEN 1 ELSE 0 END) AS converted
    FROM dwd_orders_l2 o LEFT JOIN dwd_payments_l2 p ON o.order_id=p.order_id
    WHERE o.created_at >= '{window_start}' AND o.created_at < '{window_end}'
    GROUP BY o.order_id
), refunds AS (
    SELECT SUM(refund_amount) AS refunded FROM dwd_refunds_l2
    WHERE refund_time >= '{window_start}' AND refund_time < '{window_end}'
)
SELECT CAST('{as_of}' AS DATETIME) AS as_of,
       (SELECT SUM(amount) FROM paid) AS paid_gmv,
       (SELECT COUNT(DISTINCT order_id) FROM paid) AS paid_orders,
       (SELECT COUNT(*) FROM buyer_orders) AS paid_customers,
       (SELECT COUNT(*) FROM buyer_orders WHERE paid_orders >= 2) AS repeat_customers,
       (SELECT COUNT(*) FROM cohort) AS created_orders,
       (SELECT SUM(converted) FROM cohort) AS converted_orders,
       (SELECT SUM(converted)*1.0/NULLIF(COUNT(*),0) FROM cohort) AS cohort_conversion,
       (SELECT SUM(CASE WHEN paid_orders>=2 THEN 1 ELSE 0 END)*1.0/NULLIF(COUNT(*),0) FROM buyer_orders) AS repeat_buyer_rate,
       (SELECT refunded FROM refunds) AS refund_amount,
       (SELECT SUM(amount) FROM paid)-(SELECT refunded FROM refunds) AS net_collections
""")
expect(lab.query("SELECT paid_gmv,paid_orders,paid_customers,repeat_customers,created_orders,converted_orders,refund_amount,net_collections FROM bi_paid_kpis_l2"), [("660.00",5,3,2,8,5,"50.00","610.00")],
       title="The window's contract, from the detail")
expect(lab.query("SELECT SUM(paid_customers) FROM paid_order_metrics_l2"), [(4,)], title="The daily customer counts add up to 4, not to the window's 3")
lab.sql("SELECT * FROM bi_paid_kpis_l2", title="The payment contract for a dashboard");

**Expected result**

Two green checks, then one row: as of 2026-01-04 00:00:00, paid GMV 660.00 from 5 paid orders and 3 customers, 2 of them repeat customers; 8 orders created in the window, 5 of them paid before the cutoff, a `cohort_conversion` of 0.62500; a `repeat_buyer_rate` of 0.66666 (2 of 3); 50.00 refunded; and 610.00 net collections.

The cohort counts all 8 orders created in the window, the cancelled 2107 too: it was created and not paid, so it did not convert. The order metrics of section 1 leave cancelled orders out; this contract says that it does not, and why. Both ratios keep their numerator and denominator in the row, so a reader can check them.

## 7. Exact sets and approximate sketches

An exact reconciliation cannot silently accept a HyperLogLog (HLL) estimate. Bitmap aggregation is exact for the compatible non-negative integer customer IDs used here. This generated input has 100,000 rows but only 20,000 distinct IDs; repeated IDs must not become additional customers.

Compare an exact SQL set, an exact Bitmap set, and an HLL sketch on **the same population**. The displayed relative error is one observation, not a universal error bound. The small three-customer business case would be too small to explain this trade-off.

In [ ]:
distinct_counts = lab.query("""SELECT COUNT(DISTINCT customer_id),
    BITMAP_UNION_COUNT(TO_BITMAP(customer_id)), HLL_UNION_AGG(HLL_HASH(CAST(customer_id AS STRING)))
FROM (SELECT number % 20000 + 1 AS customer_id FROM numbers("number"="100000")) ids""")[0]
expect(distinct_counts[:2], (20000,20000), title="Two exact methods agree despite repeated IDs")
from dw_course.ui import show_records
show_records("Same-population distinct counts", [
    {"method":"COUNT(DISTINCT)", "count":distinct_counts[0], "exact":True},
    {"method":"Bitmap", "count":distinct_counts[1], "exact":True},
    {"method":"HLL", "count":distinct_counts[2], "exact":False,
     "observed_relative_error":abs(distinct_counts[2]-20000)/20000},
])

**Expected result**

A green check, the two exact methods agree on 20,000, then the three counts: `COUNT(DISTINCT)` and Bitmap 20,000 (exact), and HLL a number close to it, 19,834, with an `observed_relative_error` of 0.0083, under 1 %.

HLL keeps a small sketch instead of every ID, so its count is an estimate; one observed error is not a bound for another population. Use it where an approximate count is good enough and the consumers know that it is approximate.

## 8. Measure a result-preserving optimization

Generate 100,000 detail orders across 32 dates, then rebuild a 32-row daily service table. Both queries must return **every date, count, and amount identically**. Only then compare their actual Query Profiles.

The helper displays the measured Profile IDs and scan counters. It also takes 20 same-host latency samples per query with concurrency one, one warm-up, and SQL result caches disabled. It restores all modified session variables. Storage caches are not flushed. The empirical 95th percentile (P95) is illustrative: this single-node test does not prove a dashboard service-level agreement (SLA), multi-user behavior, or production speedup.

In [ ]:
lab.execute("DROP TABLE IF EXISTS perf_orders_detail_l2")
lab.execute("DROP TABLE IF EXISTS perf_daily_orders_l2")
lab.execute("""CREATE TABLE perf_orders_detail_l2 (
    order_date DATE NOT NULL, order_id BIGINT NOT NULL, amount DECIMAL(18,2) NOT NULL
) DUPLICATE KEY(order_date,order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
lab.execute("""INSERT INTO perf_orders_detail_l2
SELECT DATE_ADD(CAST('2026-01-01' AS DATE), CAST(number%32 AS INT)), number, number%100+1
FROM numbers("number"="100000")""")
lab.execute("""CREATE TABLE perf_daily_orders_l2 (
    order_date DATE NOT NULL, order_count BIGINT SUM NOT NULL,
    amount DECIMAL(18,2) SUM NOT NULL
) AGGREGATE KEY(order_date) DISTRIBUTED BY HASH(order_date) BUCKETS 1
PROPERTIES ("replication_num"="1")""")
# Rebuild, rather than appending the same additive batch twice.
lab.execute("INSERT INTO perf_daily_orders_l2 SELECT order_date,COUNT(*),SUM(amount) FROM perf_orders_detail_l2 GROUP BY order_date")
expect(lab.query("SELECT COUNT(*),SUM(amount) FROM perf_orders_detail_l2"), [(100000,"5050000.00")])
expect(lab.query("SELECT SUM(order_count),SUM(amount) FROM perf_daily_orders_l2"), [(100000,"5050000.00")],
       title="The 32-row table holds the same 100,000 orders and 5,050,000.00")
profile_evidence = compare_query_profiles(lab, [
    ("Detail aggregation", "SELECT order_date,COUNT(*) AS orders,SUM(amount) AS amount FROM perf_orders_detail_l2 GROUP BY order_date ORDER BY order_date"),
    ("Daily service aggregation", "SELECT order_date,SUM(order_count) AS orders,SUM(amount) AS amount FROM perf_daily_orders_l2 GROUP BY order_date ORDER BY order_date"),
]);

**Expected result**

A green check, then a green check that both queries return the same 32 rows, the two complete Query Profiles (collapsed), and the scan evidence: the detail aggregation reads 100,000 rows and the service aggregation 32. Then the latency samples: 20 runs of each query, one at a time, with an empirical P95 of a few milliseconds for each; the service table is usually faster, but on this small table the difference is small and can change from run to run.

The equal results come first: a faster query that returns a different answer is not an optimization. The Profile's counters show what each execution read; the timings belong to this computer and this moment, not to a production dashboard.

## 9. Practice: orders per customer and day

The business asks for orders and order amount per customer and day. `daily_order_metrics_l2` cannot answer: its rows have no customer, and no query can bring one back. Compute the customer-day grain from `orders_fact_l2` with the same contract, show its rows, and check that they add up to the detail's 10 orders and 1310.00. Leave `daily_order_metrics_l2` as it is: Lab 11 reads it.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
customer_day_sql = """SELECT order_date, customer_id, COUNT(*) AS order_count, SUM(order_amount) AS gross_amount
FROM orders_fact_l2
WHERE status <> 'CANCELLED'
GROUP BY order_date, customer_id"""
lab.sql(customer_day_sql + " ORDER BY order_date, customer_id", title="Orders per customer and day")
expect(lab.query(f"SELECT COUNT(*), SUM(order_count), SUM(gross_amount) FROM ({customer_day_sql}) AS customer_days"),
       [(10, 10, "1310.00")], title="Ten customer-days, 10 orders, 1310.00")
expect(lab.query("SELECT SUM(order_count), SUM(gross_amount) FROM daily_order_metrics_l2"), [(10, "1310.00")],
       title="The service table is unchanged")

**Expected result**

Ten rows, one per customer and day: customers 1 and 3–10 on 2026-01-01 and customer 11 on 2026-01-02, each with one order, from 50.00 (customer 9) to 300.00 (customer 7). Two green checks: ten customer-days, 10 orders, and 1310.00, and the service table still 10 orders and 1310.00.

Each customer has one order in this data, so the customer-day grain has as many rows as the orders. A grain that the service table does not keep comes from the detail; giving every customer the day's total instead would count the day once per customer.

## You are done when

- you can state the contract of `daily_order_metrics_l2`: population, grain, measures, time, and ratio, and why it stores sums and counts;
- the service table matches the detail in a full outer join, and you can say why the inner join missed the lost day;
- you can explain why the average of the region averages, 132.00, is not January 1's average order amount, 133.33;
- you can tell the case's paid GMV, 660.00, from its order amount, and say what each ratio of `bi_paid_kpis_l2` counts;
- you can say what the equal results, the Profiles' counters, and the timings each show about the service table.

## Next

Open [Quiz 10: Metric Processing and Delivery](quiz10_metric_processing.ipynb), then continue to [Module 11: Interfaces and Boundaries for BI and AI Consumers](../module11-bi-and-ai/course.md).